# Tutoriel : Commande multi-variable linéaire

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/numpy_state_space.ipynb)

Les mêmes étapes, trois fois, sur un système masse–ressort–amortisseur ($k/m = 4$, $b/m = 1$) :

1. **NumPy / SciPy / Matplotlib** — les matrices $A,B,C,D$ à la main
2. **python-control** — `ss`, `place`, `lqr`, `lqe`
3. **minilink** — le même modèle comme un `SingleMass` (`place()` et Kalman : pas encore dans minilink)


In [ ]:
import numpy as np
import scipy.linalg as linalg
import scipy.signal as signal
import matplotlib.pyplot as plt

## Partie 1 — NumPy, SciPy et Matplotlib

Implémentation directe avec `numpy`, `scipy` et `matplotlib`.


## 1. Définition du Système

Nous commençons par définir un système d'équations d'état linéaire, invariant dans le temps (LTI) :

$$
\dot{x} = Ax + Bu
$$
$$
y = Cx + Du
$$

Nous utiliserons un exemple masse-ressort-amortisseur

In [ ]:
A = np.array([
    [0, 1],
    [-4, -1]  # Paramètres: k/m = 4, b/m = 1
])
B = np.array([
    [0],
    [1]     # La force agit sur la masse
])
C = np.array([[1, 0]])  # On mesure la position
D = np.array([[0]])

n = A.shape[0]  # Ordre du système

print(f"Matrice A:\n{A}")
print(f"Matrice B:\n{B}")
print(f"Matrice C:\n{C}")

## 2. Analyse de Base

### Pôles en Boucle Ouverte
Les pôles du système en boucle ouverte sont les valeurs propres de la matrice $A$. Ils déterminent la stabilité et le comportement naturel (fréquence, amortissement) du système.

$$
\lambda_i = \text{eig}(A)
$$

In [ ]:
poles_bo = linalg.eigvals(A)
print(f"Pôles en boucle ouverte (eig):\n{poles_bo}")
print(f"Le système est stable car la partie réelle de tous les pôles est négative:\n{poles_bo.real}")

### Contrôlabilité
Un système est contrôlable si, pour n'importe quel état initial $x(0)$ et état final $x(t_f)$, il existe une commande $u(t)$ qui amène le système de $x(0)$ à $x(t_f)$ en un temps fini.

On vérifie cela avec la matrice de contrôlabilité $W_c$ :

$$
W_c = [ B \ | \ AB \ | \ A^2B \ | \ \dots \ | \ A^{n-1}B ]
$$

Le système est contrôlable si $W_c$ est de plein rang.

$$
\text{rank}(W_c) = n
$$


---

### Observabilité
Un système est observable si, pour une sortie $y(t)$ mesurée sur un temps fini, on peut déterminer de manière unique l'état initial $x(0)$.

On vérifie cela avec la matrice d'observabilité $W_o$ :

$$
W_o = \begin{bmatrix}
C \\
CA \\
CA^2 \\
\vdots \\
CA^{n-1}
\end{bmatrix}
$$

Le système est observable si $W_o$ est de plein rang.

$$
\text{rank}(W_o) = n
$$

Ici on définit des fonctions pour construire ces matrices:

In [ ]:
def ctrb(A, B):
    """Calcule la matrice de contrôlabilité Wc."""
    n = A.shape[0]
    Wc = B
    for i in range(1, n):
        Wc = np.hstack((Wc, np.linalg.matrix_power(A, i) @ B))
    return Wc

def obsv(A, C):
    """Calcule la matrice d'observabilité Wo."""
    n = A.shape[0]
    Wo = C
    for i in range(1, n):
        Wo = np.vstack((Wo, C @ np.linalg.matrix_power(A, i)))
    return Wo

Testons maintenant ces propriétés pour notre exemple:

In [ ]:
Wc = ctrb(A, B)

rank_Wc = np.linalg.matrix_rank(Wc)
print(f"Matrice de contrôlabilité:\n{Wc}")
print(f"Rang de Wc (rank): {rank_Wc} -> {'Contrôlable' if rank_Wc == n else 'Non contrôlable'}\n")

Wo = obsv(A, C)

rank_Wo = np.linalg.matrix_rank(Wo)
print(f"Matrice d'observabilité:\n{Wo}")
print(f"Rang de Wo (rank): {rank_Wo} -> {'Observable' if rank_Wo == n else 'Non observable'}")

## 3. Conversion vers TF (Pôles-Zéros-Gain)

Si on sélection une entrée et une sortie, on peut calculer la fonction de transfer associé à partir de nos matrices A,B,C et D

La fonction de transfert $G(s)$ est donnée par :

$$
G(s) = \frac{Y(s)}{U(s)} = C(sI - A)^{-1}B + D
$$

Une fonction de transfer est pleinement déterminer par 1) ses pôles (racines du dénominateur), 2) ses zéros (racines du numérateur) and 3) le gain statique.

La fonction `scipy.signal.ss2zpk` (State-Space to Zero-Pole-Gain) permet de les extraires directements:

In [ ]:
# Conversion ss -> zpk
zeros, poles, gain = signal.ss2zpk(A, B, C, D)

print(f"Pôles: {poles}")
print(f"Zéros: {zeros}")
print(f"Gain: {gain}")

Note: les pôles sont en fait toujorus exactement les valeurs propres de la matrice A:

In [ ]:
poles_bo = linalg.eigvals(A)
print(f"Pôles de la tf : {poles}")
print(f"Valeurs propres: {poles_bo}")

On peut visualiser la fonction de transfer avec un diagramme des pôles et zéros sur le plan complex:

In [ ]:
plt.figure()
# Pôles
plt.scatter(np.real(poles), np.imag(poles), marker='x', color='r', s=100, label='Pôles')
# Zéros
if zeros.size > 0:
    plt.scatter(np.real(zeros), np.imag(zeros), marker='o', color='b', s=100, facecolors='none', label='Zéros')

plt.title('Pôle-Zéro Map (pzmap)')
plt.xlabel('Axe Réel')
plt.ylabel('Axe Imaginaire')
plt.axhline(0, color='k', lw=0.5)
plt.axvline(0, color='k', lw=0.5)
plt.grid(True)
plt.legend()
plt.axis('equal')
plt.show()

## 4. Commande par Placement de Pôles (`place`)

Si le système est contrôlable, nous pouvons utiliser un retour d'état $u = -Kx$ pour placer les pôles en boucle fermée (les valeurs propres de $A_{cl}$) à n'importe quel endroit désiré dans le plan complexe.

Le système en boucle fermée devient :

$$
\dot{x} = Ax + B(-Kx) = (A - BK)x
$$

Nous cherchons $K$ tel que $\text{eig}(A - BK) = \{ p_1, p_2, \dots, p_n \}_{\text{désirés}}$.

`scipy.signal.place_poles` implémente l'algorithme d'Ackermann pour cela.

In [ ]:
# Pôles désirés (plus rapides et plus amortis que les pôles BO)
poles_desires = np.array([-3.0, -3.5])
print(f"Pôles désirés: {poles_desires}\n")

result = signal.place_poles(A, B, poles_desires)
K = result.gain_matrix
print(f"Gain de retour d'état K (place):\n{K}\n")

On peut recalculer les pôles du système en boucle fermée pour vérifier:

In [ ]:
# Vérification
A_cl = A - B @ K
poles_cl = linalg.eigvals(A_cl)
print(f"Pôles réels en boucle fermée (eig(A-BK)): {np.sort(poles_cl)}")

## 5. Commande Optimale (`lqr`)

Le régulateur linéaire quadratique (LQR) trouve le gain de retour d'état $K$ qui minimise une fonction de coût quadratique :

$$
J = \int_{0}^{\infty} (x^T Q x + u^T R u) \,dt
$$

Où $Q \ge 0$ pénalise l'écart des états et $R > 0$ pénalise l'effort de commande.

Utilisons les valeurs suivantes pour notre exemple:

In [ ]:
# Matrices de pondération
Q = np.diag([1.0, 1.0])  # Pénalité sur les états x'Qx
R = np.array([[0.1]])    # Pénalité sur la commande u'Ru

print(f"Matrice Q:\n{Q}")
print(f"Matrice R:\n{R}\n")

La solution est sous la forme:

$$
J = x^T S x
$$

ou $S$ est trouvé en résolvant l'équation de Riccati (CARE) pour $P$:

$$
A^T S + SA - S B R^{-1} B^T S + Q = 0
$$

La fonction `scipy.linalg.solve_continuous_are` résout l'équation de Riccati numériquement:




In [ ]:
S = linalg.solve_continuous_are(A, B, Q, R)
print(f"Solution de Riccati S :\n{S}\n")

Le gain optimal est alors donné par :

$$
K = R^{-1} B^T S
$$


In [ ]:
R_inv = linalg.inv(R)
K_lqr = R_inv @ B.T @ S
print(f"Gain LQR K_lqr:\n{K_lqr}\n")

On peut aussi vérifier les valeurs propres en boucle fermée:

In [ ]:
# Vérification
A_lqr = A - B @ K_lqr
poles_lqr = linalg.eigvals(A_lqr)
print(f"Pôles LQR en boucle fermée (eig(A-BK)):\n{poles_lqr}")

## 6. Filtre de Kalman (`kalman`)

Le filtre de Kalman est un observateur optimal pour un système bruité. On considère le système :

$$
\dot{x} = Ax + Bu + w \quad (w \sim N(0, Q_n))
$$
$$
y = Cx + Du + v \quad (v \sim N(0, R_n))
$$

Où $Q_n$ est la covariance du bruit de processus et $R_n$ est la covariance du bruit de mesure.

L'observateur (le filtre) a la dynamique suivante, où $L$ est le gain de Kalman :

$$
\dot{\hat{x}} = A\hat{x} + Bu + L(y - \hat{y}) \quad \text{où } \hat{y} = C\hat{x} + Du
$$

Le gain $L$ est trouvé en résolvant l'équation de Riccati (CARE) *duale* pour la covariance de l'erreur d'estimation $P_e$:

$$
A P_e + P_e A^T - P_e C^T R_n^{-1} C P_e + Q_n = 0
$$

Le gain de Kalman est alors :

$$
L = P_e C^T R_n^{-1}
$$

**Astuce :** On peut utiliser le même solveur `solve_continuous_are` en utilisant la dualité de LQR ($A \to A^T$, $B \to C^T$, $Q \to Q_n$, $R \to R_n$).

In [ ]:
# Covariances des bruits
Qn = np.diag([0.1, 0.1]) # Bruit de processus w
Rn = np.array([[0.1]])   # Bruit de mesure v

print(f"Covariance Qn (processus):\n{Qn}")
print(f"Covariance Rn (mesure):\n{Rn}\n")

# 1. Résoudre la CARE duale (transposer A et C)
# Note : A -> A.T, B -> C.T, Q -> Qn, R -> Rn
Pe = linalg.solve_continuous_are(A.T, C.T, Qn, Rn)
print(f"Covariance d'erreur Pe (P en MATLAB):\n{Pe}\n")

# 2. Calculer le gain L
Rn_inv = linalg.inv(Rn)
L_kalman = Pe @ C.T @ Rn_inv
print(f"Gain de Kalman L:\n{L_kalman}\n")

# Vérification des pôles de l'estimateur
A_est = A - L_kalman @ C
poles_est = linalg.eigvals(A_est)
print(f"Pôles de l'estimateur (eig(A-LC)):\n{poles_est}")

## Partie 2 — python-control

La bibliothèque `python-control` (`pip install control`) fournit une API de haut niveau.

https://python-control.readthedocs.io/en/0.10.2/

Les commandes équivalentes pour le même système ($A,B,C,D$ de la partie 1).


In [ ]:
import importlib.util

if importlib.util.find_spec("control") is None:
    get_ipython().system("pip install -q control")


In [ ]:
import control as ct
import numpy as np

In [ ]:
# On crée un objet "StateSpace" (ss)
sys = ct.ss(A, B, C, D)
print(f"Système (python-control):\n{sys}")

# Pôles (eig)
poles_sys = sys.poles()
print(f"\nPôles (eig): {poles_sys}")

### Controllabilité / Observabilité

In [ ]:

Wc_ct = ct.ctrb(A, B)

print(f"\nMatrice de contrôlabilité (ctrb):\n{Wc_ct}")
print(f"Rang (rank): {np.linalg.matrix_rank(Wc_ct)}")

Wo_ct = ct.obsv(A, C)

print(f"\nMatrice d'observabilité (obsv):\n{Wo_ct}")
print(f"Rang (rank): {np.linalg.matrix_rank(Wo_ct)}")

In [ ]:
ct.pzmap(sys, plot=True, title="PZ Map (python-control)")

### Placement de pôles

In [ ]:

poles_desires = np.array([-3.0, -3.5])
K_place_ct = ct.place(A, B, poles_desires)
print(f"\nGain K (place):\n{K_place_ct}")

### LQR

In [ ]:
K_lqr_ct, S_lqr_ct, E_lqr_ct = ct.lqr(A, B, Q, R)
print(f"\nGain K (lqr):\n{K_lqr_ct}")


### Filtre de Kalman

https://python-control.readthedocs.io/en/0.10.2/generated/control.lqe.html#control.lqe

In [ ]:
G  = np.eye(n)  # Matrice de bruit de processus

L_kalman_ct, Pe_kalman_ct, E_kalman_ct = ct.lqe(A, G, C, Qn, Rn)

print(f"\nGain L (kalman):\n{L_kalman_ct}")

## Partie 3 — minilink

Le même exemple, avec [minilink](https://github.com/alx87grd/minilink) : `SingleMass(mass=1, k=4, b=1)` a les mêmes matrices $A,B,C,D$ que la partie 1.

`place()` et le filtre de Kalman ne sont pas encore dans minilink ; pour ces deux étapes, rester sur la partie 1 ou 2.


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
from minilink import (
    SingleMass,
    controllability,
    lqr,
    observability,
    plot_pzmap,
    pzmap,
    transfer_function,
)


### Définition du système


In [ ]:
plant = SingleMass(mass=1.0, k=4.0, b=1.0)

A, B, C, D = plant.A(), plant.B(), plant.C(), plant.D()

print(f"Matrice A:\n{A}")
print(f"Matrice B:\n{B}")
print(f"Matrice C:\n{C}")


### Pôles, contrôlabilité, observabilité


In [ ]:
zeros, poles, gain = pzmap(plant)
print(f"Pôles (pzmap): {poles}")
print(f"Zéros: {zeros}")
print(f"Gain: {gain}")

Wc = controllability(plant)
print(f"\nMatrice de contrôlabilité:\n{Wc.matrix}")
print(f"Rang de Wc: {Wc.rank} -> {'Contrôlable' if Wc.is_full_rank else 'Non contrôlable'}")

Wo = observability(plant)
print(f"\nMatrice d'observabilité:\n{Wo.matrix}")
print(f"Rang de Wo: {Wo.rank} -> {'Observable' if Wo.is_full_rank else 'Non observable'}")


### Conversion vers TF et carte pôles–zéros


In [ ]:
G = transfer_function(plant)
print(G.name, "num:", G.numerator, "den:", G.denominator)


In [ ]:
plot_pzmap(plant)


### Placement de pôles

`place()` n'est pas encore dans minilink. Voir la partie 1 (`scipy.signal.place_poles`) ou la partie 2 (`control.place`).


### LQR


In [ ]:
Q = np.diag([1.0, 1.0])
R = np.array([[0.1]])

ctl = lqr(A, B, Q, R)
K_lqr = ctl.params["K"]
print(f"Gain LQR K:\n{K_lqr}")

A_lqr = A - B @ K_lqr
poles_lqr = np.linalg.eigvals(A_lqr)
print(f"\nPôles LQR en boucle fermée (eig(A-BK)):\n{poles_lqr}")


### Filtre de Kalman

Le bloc Kalman (`estimation/`) n'est pas encore dans minilink. Voir la partie 1 (Riccati duale) ou la partie 2 (`control.lqe`).
